In [0]:
# Cell 1: imports, settings and city config
import os
import gzip
import math
import requests
import yaml
import xml.etree.ElementTree as ET
from datetime import date, datetime, timezone

CATALOG = "city_delta"
USER_AGENT = "CityDelta/1.0 (portfolio project)"
BRONZE_TABLE = f"{CATALOG}.bronze.ndw_raw"
NDW_URL = "https://opendata.ndw.nu/planningsfeed_wegwerkzaamheden_en_evenementen.xml.gz"

with open("../config/cities.yaml") as f:
    cities = yaml.safe_load(f)["cities"]

today = date.today()
ingest_ts = datetime.now(timezone.utc)

def local(tag):
    """'{namespace}situation' -> 'situation'"""
    return tag.split("}")[-1]

In [0]:
# Cell 2: download the national file once and archive it
out_dir = f"/Volumes/{CATALOG}/bronze/raw/ndw/{today.isoformat()}"
os.makedirs(out_dir, exist_ok=True)
gz_file = f"{out_dir}/planningsfeed_wegwerkzaamheden_en_evenementen.xml.gz"

with requests.get(NDW_URL, headers={"User-Agent": USER_AGENT}, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    source_last_modified = resp.headers.get("Last-Modified")
    with open(gz_file, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):   # 1 MB at a time
            f.write(chunk)

print("Saved:", gz_file)
print("Size MB:", round(os.path.getsize(gz_file) / 1e6, 1))
print("Source last modified:", source_last_modified)

In [0]:
# Cell 4: keep only situations inside each city's bounding box
def bounding_box(city):
    """Square box around the centre: radius_km in every direction."""
    r = city["radius_km"]
    dlat = r / 111.0                                         # ~111 km per degree of latitude
    dlon = r / (111.32 * math.cos(math.radians(city["lat"])))  # degrees of longitude shrink with latitude
    return (city["lat"] - dlat, city["lat"] + dlat,
            city["lon"] - dlon, city["lon"] + dlon)

def coordinates(situation):
    """All (lat, lon) pairs inside one situation."""
    lats = [float(e.text) for e in situation.iter() if local(e.tag) == "latitude"]
    lons = [float(e.text) for e in situation.iter() if local(e.tag) == "longitude"]
    pairs = list(zip(lats, lons))
    # Line locations can use a posList: "a b a b ..." pairs
    for e in situation.iter():
        if local(e.tag) == "posList" and e.text:
            nums = [float(x) for x in e.text.split()]
            for a, b in zip(nums[0::2], nums[1::2]):
                pairs.append((max(a, b), min(a, b)))  # in NL latitude (50–54) is always > longitude (3–7)
    return pairs

boxes = {c["key"]: bounding_box(c) for c in cities}
rows = []

with gzip.open(gz_file, "rb") as f:
    for event, elem in ET.iterparse(f, events=("end",)):
        if local(elem.tag) != "situation":
            continue
        pairs = coordinates(elem)
        for city_key, (lat_min, lat_max, lon_min, lon_max) in boxes.items():
            if any(lat_min <= lat <= lat_max and lon_min <= lon <= lon_max for lat, lon in pairs):
                rows.append({
                    "city_key": city_key,
                    "situation_id": elem.get("id"),
                    "ingest_ts": ingest_ts,
                    "ingest_date": today,
                    "source_last_modified": source_last_modified,
                    "raw_file": gz_file,
                    "raw_xml": ET.tostring(elem, encoding="unicode"),
                })
        elem.clear()

print("Situations kept:", len(rows))
for key, box in boxes.items():
    print(key, [round(x, 4) for x in box])

In [0]:
# Cell 5: write to the bronze Delta table (idempotent per day) and check
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType

schema = StructType([
    StructField("city_key",             StringType(),    False),
    StructField("situation_id",         StringType(),    True),
    StructField("ingest_ts",            TimestampType(), False),
    StructField("ingest_date",          DateType(),      False),
    StructField("source_last_modified", StringType(),    True),
    StructField("raw_file",             StringType(),    True),
    StructField("raw_xml",              StringType(),    False),
])

df = spark.createDataFrame(rows, schema=schema)

if spark.catalog.tableExists(BRONZE_TABLE):
    spark.sql(f"DELETE FROM {BRONZE_TABLE} WHERE ingest_date = DATE'{today.isoformat()}'")

df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

display(spark.sql(f"""
    SELECT ingest_date, city_key, count(*) AS situations, count(DISTINCT situation_id) AS unique_ids
    FROM {BRONZE_TABLE}
    GROUP BY ingest_date, city_key
"""))